# COMPSCI5094 Coursework - Conversational Interfaces: Premier League Dialogue Agent
**Author**: Patricia Natasha Munginga

**Student ID**: 3033388M

**Date**: February 2026

**Project**: Task‑Oriented Dialogue System for Premier League Information

This notebook documents the development of a task‑oriented conversational agent that provides information about English Premier League teams and personnel using a football data API and an LLM‑based NLU component.
The focus is on defining intents and slots, implementing robust intent detection and slot filling with a local LLM, integrating real‑time football data via an external API and demonstrating end‑to‑end dialogues that satisfy user queries while enabling clear evaluation of NLU performance.

### Setup and Imports

In [1]:
"""Installs and Imports"""

# Install the Ollama python library
%pip install ollama

# Import the Ollama python library
import ollama

# API functionality
import requests
import json

Note: you may need to restart the kernel to use updated packages.


### Configuration

In [ ]:
# Create a client connected to the local Ollama server
client = ollama.Client(host= 'http://makatea.dcs.gla.ac.uk:11434') # Using University of Glasgow VPN for faster execution, otherwise use: 'http://localhost:11434'

# Ensure the required model is available locally (downloads it if needed)
!ollama pull qwen3:4b-instruct
!ollama pull phi3:mini

# Define the model name to use throughout the notebook
MODEL = 'qwen3:4b-instruct'  
#MODEL = "phi3:mini"

# Database and API 
FOOTBALL_DATA_API_KEY = 'YOUR_API_KEY_HERE'
FOOTBALL_DATA_BASE_URL = "https://api.football-data.org/v4"
PREMIER_LEAGUE_CODE = "PL" # English Premier League

### API Functions

In [3]:
def call_football_data_api(endpoint, params=None, verbose=True):
    """
    Generic helper function to call the football-data API.

    endpoint: string like 'teams/57'
    params: dictionary of query parameters (e.g., {'status': 'SCHEDULED'})
    verbose: if True, prints basic debug info
    """
    headers = {'X-Auth-Token': FOOTBALL_DATA_API_KEY}
    url = f"{FOOTBALL_DATA_BASE_URL}/{endpoint}"

    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)

        if verbose:
            print(f"[API] GET {url} params={params} status={response.status_code}")

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        # Return a structured error (better for tool outputs)
        return {"error": str(e), "endpoint": endpoint, "params": params}

def filter_premier_league_matches(matches):
    """
    Premier League-only filter

    Purpose: /teams/{id}/matches may include PL + other competitions (e.g., CL).
    This system is Premier League only
    """
    return [m for m in matches if m.get('competition', {}).get('code') == PREMIER_LEAGUE_CODE]    
      
def get_team_id_by_name(team_name):
    """
    Helper function to get team ID from team name

    SLOT: team (required)
    Endpoint used: /v4/competitions/PL/teams
    Purpose: Resolve team name to team ID
    """
    result = call_football_data_api(f'competitions/{PREMIER_LEAGUE_CODE}/teams', verbose = False)
    
    if not result or 'error' in result:
        return None
    
    teams = result.get('teams', [])
    team_name_lower = team_name.lower().strip()
    
    # 1) Exact match on full or short name
    for team in teams:
        if team['name'].lower() == team_name_lower or \
           team['shortName'].lower() == team_name_lower:
            return team['id']
    
    # 2) Exact match on TLA (3-letter code)
    for team in teams:
        if team['tla'].lower() == team_name_lower:
            return team['id']
    
    # 3) Partial match fallback
    for team in teams:
        if team_name_lower in team['name'].lower():
            return team['id']
    
    return None 

def get_manager_info(team_name):
    """
    Get information about a team's manager/coach

    SLOT: manager
    Endpoint used: /v4/teams/{id}
    Purpose: Returns detailed team info including coach/manager field
    """
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}
    
    result = call_football_data_api(f'teams/{team_id}', verbose = False)

    if not result or 'error' in result:
        return result if result else {"error": "API call failed"}
    
    coach = result.get('coach')

    if not coach:
        return {"error": "Manager information not available"}
    
    return {
        'manager': coach.get('name', 'N/A'),
        'nationality': coach.get('nationality', 'N/A'),
        'team': result.get('name', 'N/A')
    }

def get_standings_info(team_name):
    """
    Get standings for a team

    SLOTS:
      - leaguePosition
      - numGamesPlayed
      - winLossRecord

    Endpoint used: /v4/competitions/PL/standings
    Purpose: Returns league table including position, games played, wins, draws, losses
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Call Premier League standings endpoint
    result = call_football_data_api(
        f'competitions/{PREMIER_LEAGUE_CODE}/standings',
        verbose = False
    )

    if not result or 'standings' not in result:
        return {"error": "Standings data not available"}

    # Step 3: Find the TOTAL standings table
    for standing_type in result['standings']:
        if standing_type['type'] == 'TOTAL':

            # Step 4: Find this team in the league table
           for team_row in standing_type['table']:
                if team_name.lower() in team_row['team']['name'].lower():

                    return {
                        "team": team_row['team']['name'],
                        "leaguePosition": team_row['position'],
                        "numGamesPlayed": team_row['playedGames'],
                        "winLossRecord": f"{team_row['won']}-{team_row['draw']}-{team_row['lost']}"
                    }

    return {"error": "Team not found in standings"}

def get_next_match(team_name):
    """
    Get next match for a team

    SLOTS:
       - nextGameDate
       - nextOpponent

    Endpoint used: /v4/teams/{id}/matches?status=SCHEDULED
    Purpose: Returns upcoming matches
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get scheduled matches
    result = call_football_data_api(
        f'teams/{team_id}/matches',
        {'status': 'SCHEDULED'},
        verbose = False
    )

    if not result or 'matches' not in result:
        return {"error": "Match data not available"}

    matches = result['matches']

    # Step 3: Filter Premier League matches only
    pl_matches = filter_premier_league_matches(matches)

    if not pl_matches:
        return {"error": "No upcoming Premier League matches found"}

    # Step 4: Sort by utcDate ascending → next match first
    pl_matches = sorted(pl_matches, key=lambda m: m['utcDate'])
    next_match = pl_matches[0]

    # Step 5: Determine opponent
    if next_match['homeTeam']['id'] == team_id:
        opponent = next_match['awayTeam']['name']
    else:
        opponent = next_match['homeTeam']['name']

    return {
        "team": team_name,
        "nextGameDate": next_match['utcDate'],
        "nextOpponent": opponent
    }


def get_last_match(team_name):
    """
    Get last played match for a team

    SLOTS:
     - lastOpponent
     - lastScore

    Endpoint used: /v4/teams/{id}/matches?status=FINISHED
    Purpose: Returns completed matches including full-time score
    """

    # Step 1: Resolve team name → team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get finished matches
    result = call_football_data_api(
        f"teams/{team_id}/matches",
        {"status": "FINISHED"},
        verbose = False
    )

    if not result or "matches" not in result:
        return {"error": "Match data not available"}

    matches = result["matches"]

    # Step 3: Filter Premier League matches only
    pl_matches = filter_premier_league_matches(matches)

    if not pl_matches:
        return {"error": "No finished Premier League matches found"}

    # Step 4: Sort by utcDate descending → most recent finished match first
    pl_matches = sorted(pl_matches, key=lambda m: m["utcDate"], reverse=True)
    last_match = pl_matches[0]

    # Step 5: Determine opponent
    if last_match["homeTeam"]["id"] == team_id:
        opponent = last_match["awayTeam"]["name"]
    else:
        opponent = last_match["homeTeam"]["name"]

    # Step 6: Extract score
    full_time = last_match["score"]["fullTime"]

    if last_match["homeTeam"]["id"] == team_id:
        team_goals = full_time["home"]
        opponent_goals = full_time["away"]
    else:
        team_goals = full_time["away"]
        opponent_goals = full_time["home"]

    last_score = f"{team_goals} - {opponent_goals}"  # always team first

    return {
        "team": team_name,
        "lastOpponent": opponent,
        "lastScore": last_score,
        "lastMatchDate": last_match["utcDate"] 
    }


def get_playing_now(team_name):
    """
    Check if a team is currently playing a match
    
    SLOT: playingNow

    Endpoint used: /v4/teams/{id}/matches?status=IN_PLAY
    Purpose: Checks if team currently has a live match
    """

    # Step 1: Resolve team name - team ID
    team_id = get_team_id_by_name(team_name)

    if not team_id:
        return {"error": f"Team '{team_name}' not found in Premier League"}

    # Step 2: Get live matches
    result = call_football_data_api(
        f"teams/{team_id}/matches",
        {"status": "IN_PLAY"},
        verbose = False
    )

    if not result or "matches" not in result:
        return {"playingNow": False}

    matches = result["matches"]

    # Step 3: Filter Premier League matches only and Check if any PL match is live
    if not result or 'error' in result or "matches" not in result:
        return {"team": team_name, "playingNow": False}

    pl_matches = filter_premier_league_matches(result["matches"])

    return {
        "team": team_name,
        "playingNow": len(pl_matches) > 0
    }

### Dialogue Manager

In [4]:
"""
DIALOGUE MANAGER

Responsible for:
1. Receiving tool calls from the LLM (intent + slot extraction).
2. Executing the corresponding backend function.
3. Returning structured API results back to the LLM.
4. Generating the final natural language response.

This separates:
- NLU (model selects tool)
- Backend execution (Python functions + API)
- NLG (model generates final response)

Ensures deterministic, tool-grounded behaviour.
"""
import re

def generate_response(result, requested_slots, team_name):

    if "error" in result:
        return result["error"]

    nlg_prompt = [
        {
            "role": "system",
            "content": """You are a Premier League football assistant. Convert structured data into a single natural, conversational sentence.

RULES:
- Only use information present in the data provided.
- For scores, always state the outcome (won/lost/drew), the score, and the opponent.
- Keep responses concise — one sentence only.
- Do not add information not in the data.

EXAMPLES:
Data: {"team": "Manchester City", "lastOpponent": "Bristol City", "lastScore": "3 - 2", "lastMatchDate": "2024-02-15T00:00:00Z"}
Slots: ["lastScore"]
Response: Manchester City won with a score of 3-2 against Bristol City.

Data: {"team": "Burnley", "lastOpponent": "Chelsea", "lastScore": "3 - 2", "lastMatchDate": "2024-08-12T00:00:00Z"}
Slots: ["lastScore"]
Response: They played their last game on August 12th against Chelsea and they won 3-2.

Data: {"team": "Huddersfield", "lastOpponent": "Stoke City", "lastScore": "0 - 2", "lastMatchDate": "2024-01-20T00:00:00Z"}
Slots: ["lastScore"]
Response: They lost 2 to 0 against Stoke City on January 20th.

Data: {"team": "Everton", "leaguePosition": 15}
Slots: ["leaguePosition"]
Response: Everton is number 15 in the English Premier League standings.

Data: {"team": "Burnley", "nextOpponent": "West Brom", "nextGameDate": "2024-08-19T00:00:00Z"}
Slots: ["nextOpponent"]
Response: Burnley will play against West Brom on August 19th.

Data: {"team": "Manchester City", "manager": "Pep Guardiola"}
Slots: ["manager"]
Response: Pep Guardiola.

Data: {"team": "Manchester City", "winLossRecord": "21-2-1"}
Slots: ["winLossRecord"]
Response: 21 wins, 2 draws and 1 loss.

Data: {"team": "Everton", "playingNow": false}
Slots: ["playingNow"]
Response: Everton was not scheduled to play today.

Data: {"team": "Everton", "lastOpponent": "Watford", "lastMatchDate": "2024-11-05T00:00:00Z"}
Slots: ["lastOpponent"]
Response: Everton played Watford on November 5th.

Data: {"team": "Burnley", "numGamesPlayed": 1}
Slots: ["numGamesPlayed"]
Response: They have played just one match.

Output the response sentence only. No explanation, no preamble."""
        },
        {
            "role": "user",
            "content": f"Data: {json.dumps(result)} Slots: {requested_slots}"
        }
    ]

    response = client.chat(
        model=MODEL,
        messages=nlg_prompt,
        options={"temperature": 0.3},
    )

    raw = response["message"]["content"]
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    return raw


### Main Loop

In [ ]:
FUNCTION_TO_SLOTS = {
    "get_manager_info": ["manager"],
    "get_standings_info": ["leaguePosition", "numGamesPlayed", "winLossRecord"],
    "get_next_match": ["nextGameDate", "nextOpponent"],
    "get_last_match": ["lastOpponent", "lastScore"],
    "get_playing_now": ["playingNow"]
}

SLOT_TO_FUNCTION = {
    "manager": "get_manager_info",
    "leaguePosition": "get_standings_info",
    "numGamesPlayed": "get_standings_info",
    "winLossRecord": "get_standings_info",
    "nextGameDate": "get_next_match",
    "nextOpponent": "get_next_match",
    "lastOpponent": "get_last_match",
    "lastScore": "get_last_match",
    "playingNow": "get_playing_now",
}

def run_football_agent():

    print("ASSISTANT: Hi, how can I help you?\n")

    MAX_TURNS = 20
    turn_count = 0
    last_team = None

    while turn_count < MAX_TURNS:

        user_message = input("USER: ").strip()

        if not user_message:
            continue

        print(f"USER: {user_message}")

        # NLU STEP (Intent + Slots as JSON)

        nlu_prompt = [
            {
                "role": "system",
                "content": """
        You are a Premier League football assistant.

        Extract structured information from the user request.

        Return ONLY valid JSON:

        {
        "intent": "GetInfo or Other",
        "function": "get_manager_info | get_standings_info | get_next_match | get_last_match | get_playing_now | null",
        "team_name": "string or null",
        "requested_slots": []
        }

        VALID SLOT NAMES:

        lastOpponent, lastScore, leaguePosition, manager,
        nextGameDate, nextOpponent, numGamesPlayed,
        playingNow, winLossRecord

        RULES:

        - If the user asks for "record", "win-loss record", "wins and losses", or "form" → ["winLossRecord"]
        - If the user asks for standings generally → include: ["leaguePosition","numGamesPlayed","winLossRecord"]
        - If the user asks for  "information" → include: ["leaguePosition", "winLossRecord"]
        - If asking who they played last → ["lastOpponent"]
        - If asking score of last game or how they did in their last game → ["lastScore"]
        - If asking how they did in last game → ["lastScore"]
        - If asking next match date → ["nextGameDate"]
        - If asking next opponent → ["nextOpponent"]
        - If asking manager → ["manager"]
        - If asking if playing now → ["playingNow"]
        - If the user corrects themselves mid-message (e.g. "sorry", "I mean", "actually", "wait") → only extract the intent from the LAST question they ask, ignore the earlier one
        - If the user makes a general statement or greeting without asking about a specific team or slot (e.g. "I have some questions", "I need some help", "hello") → intent = "Other"
        - If the user says , "thanks", "thank you", "that's all", "bye", "goodbye", or similar closing phrases → intent = "Other"
        """
            },
            {"role": "user", "content": user_message}
        ]

        response = client.chat(
            model=MODEL,
            messages=nlu_prompt,
            options={"temperature": 0},
        )

        import json

        raw_nlu = response["message"]["content"]
        raw_nlu = re.sub(r"<think>.*?</think>", "", raw_nlu, flags=re.DOTALL).strip()

        try:
            parsed = json.loads(raw_nlu)
        except:
            print("ASSISTANT: Sorry, I didn't understand.")
            continue

        intent = parsed["intent"]
        function_name = parsed["function"]
        team_name = parsed["team_name"]

        requested_slots = parsed.get("requested_slots", [])

        # Always trust slots over function name — slots are more reliably extracted
        if requested_slots:
            function_name = SLOT_TO_FUNCTION.get(requested_slots[0], function_name)

        # Safety rule: if no function detected, treat as Other
        if intent == "GetInfo" and not function_name:
            intent = "Other"

        # Pronoun memory support
        if team_name:
            last_team = team_name
        else:
            team_name = last_team

        slot_list = requested_slots or FUNCTION_TO_SLOTS.get(function_name, [])

        if intent == "Other":
                print("- Intent: Other")
                # Check if it's a closing phrase or just a general opener
                closing_words = ["bye", "goodbye", "thanks", "thank you", "that's all"]
                if any(word in user_message.lower() for word in closing_words):
                    print("\nASSISTANT: Goodbye.\n")
                    break
                else:
                    print("\nASSISTANT: Of course! Which team would you like to know about?\n")
                    continue
        else:
            print("- Intent: GetInfo")
            if len(slot_list) == 1:
                print(f"- Slots: {{team: '{team_name}', info: '{slot_list[0]}'}}")
            else:
                print(f"- Slots: {{team: '{team_name}', slots: {slot_list}}}")
            print()

        if not team_name:
            print("ASSISTANT: Which team are you asking about?")
            continue

        # ROUTING

        if function_name == "get_manager_info":
            result = get_manager_info(team_name)
        elif function_name == "get_standings_info":
            result = get_standings_info(team_name)
        elif function_name == "get_next_match":
            result = get_next_match(team_name)
        elif function_name == "get_last_match":
            result = get_last_match(team_name)
        elif function_name == "get_playing_now":
            result = get_playing_now(team_name)
        else:
            print("ASSISTANT: Sorry, I can't help with that.")
            continue

        # NLG STEP — LLM generates natural language from structured result
        assistant_message = generate_response(result, slot_list, team_name)

        print(f"ASSISTANT: {assistant_message}\n")

        turn_count += 1

if __name__ == "__main__":
    run_football_agent()
